# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

# 配備

**この PC の `server/` を本番へ置く。** 配備先は本番しかない。だから**下見と検査を先に通す。**

> 使い方は [00-start.ipynb](00-start.ipynb)。**まず下のセルを1回実行する。**

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない。迷ったらここから |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する。本番には触れない |
| 🔴 | **本番が変わる。** 実行前に `yes` の入力を求める |
| 🔑 | **別の窓で開く。** sudo のパスワードなど対話が要るもの |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()

## 1. 何が送られ、何が送られないか

| 送る | 送らない(**ホスト側が正本**) |
|---|---|
| `src/`(`vendor` / `uploads` / `cache` を除く) | `.env`(秘密) |
| `compose.yaml` / `compose.vps.yaml` | `src/config/*.local.php`(DB 接続・解除パスワードのハッシュ等) |
| `docker/` / `nginx/` | `nginx/km/allow-admin-home.local.conf`(個人の固定 IP) |
| `scripts/` のうち **ホストに置く6本**(check-updates / host-updates-setup / host-security-check / host-emergency / host-backup / send-log) | `scripts/*.ps1`、`docs/`、`Old/`、`backups/` |

**削除はしない。** アーカイブに無いファイルはホストに残る(消したファイルは手で消す)。
転送の前に「入っているべきもの」「入ってはいけないもの」をスクリプト自身が検査し、合わなければ送らない。

## 2. いつものの流れ

1. `check.php` を通す([01-daily-check](01-daily-check.ipynb))
2. 下見(`-WhatIfOnly`)
3. 配備。**済むと後片付け(`host-setup.ps1 -Fix`)が自動で走る**
4. 画面で確かめる

| `-Action` | いつ使うか |
|---|---|
| `deploy`(既定) | ファイルを置くだけ。**コンテナには触れない**。PHP だけ直したときはこれで効く(`src/` は bind mount) |
| `up` | `compose.yaml` を変えたとき。置いたあと `docker compose up -d` |
| `restart -Services <名前>` | nginx の設定だけ変えたときなど |

### 下見

接続もしない。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
.\deploy-to-host.ps1 -WhatIfOnly

### 配備する(ファイルを置くだけ)

ノートブックでは確認の入力ができないので、`yes` をこのセルの入力欄で受けてから `-Yes` を渡す。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "本番へファイルを配備します(コンテナには触れません)"
.\deploy-to-host.ps1 -Action deploy -Yes

### 配備して docker compose up -d

compose.yaml を変えたとき。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "本番へ配備して docker compose up -d します(変わったコンテナが作り直されます)"
.\deploy-to-host.ps1 -Action up -Yes

### nginx の設定だけ入れ替える

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "本番へ配備して reverse-proxy を再起動します(一瞬サイトが切れます)"
.\deploy-to-host.ps1 -Action restart -Services reverse-proxy -Yes

### 配備して、そのまま控えも取る

**取るのは配備の「あと」。** 先に取ると控えが配備前のものになり、配備で壊れたときに戻す先が1つ古くなる。
後片付けが通ったときだけ取る。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "本番へ配備して up -d し、そのあと控えを取ります"
.\deploy-to-host.ps1 -Action up -Backup -Yes

### 終了コード

| コード | 意味 |
|---|---|
| `0` | 全部成功 |
| `2` | 配備は済んだが、**後片付けに気になる点**がある(出力の「まとめ」を読む) |
| `3` | 配備は済んだが、**バックアップが取れなかった** |
| `1` | 転送の失敗など |

## 3. 後片付け(host-setup.ps1)

配備は所有権に触れない。一部だけは **www-data の持ち物でないと動かない**(`config/*.local.php` / `uploads/` / `cache/`)。
配備のあとに自動で `-Fix` が走るので、普段は手で叩かない。**引数なしなら調べるだけ。**

### 調べるだけ

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
.\host-setup.ps1

### 直す(所有権と権限)

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "本番の所有権と権限を直します"
.\host-setup.ps1 -Fix

### 直して up -d まで

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "本番の所有権と権限を直し、docker compose up -d します"
.\host-setup.ps1 -Fix -Up

### composer の依存を入れ直す

`composer.json` / `composer.lock` を変えたとき。**`docker compose exec web composer install` は通らない**(web の像に zip も unzip も無い)ので、使い捨ての composer コンテナで入れる。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "本番の src/vendor を入れ直します" --timeout 900
docker run --rm -v /opt/kosenmap/src:/app -u "$(id -u):$(id -g)" composer:2 install --no-dev --no-interaction

## 4. 困ったとき

| 出たもの | 原因と打つ手 |
|---|---|
| `ホストの docker compose が .env を読めません` | `.env` の値に `"` `'` `\` `$` が入っている。**英数字だけの値にする**。エラー文には値の断片が載るので、人に貼らない |
| `Host key verification failed.` / 「known_hosts にありません」 | 初めて繋ぐホスト。`-AcceptHostKey` を付ける(指紋を表示してから登録する) |
| `tar: … Cannot open: File exists` | Windows のフォルダの「読み取り専用」がモード 555 として載った。**いまは展開の前後で自動的に直す**(2026-09-13)。古い `deploy-to-host.ps1` を使っていないか |
| 転送のあと何も出ずに止まる | 以前はエラー出力でパイプが詰まっていた。**いまは5分で打ち切ってエラーを出す**。出たエラーの頭30行を読む |
| `リモートスクリプトが … 秒で終わらなかった` | ホストが重い / 回線が切れた。[06-emergency](06-emergency.ipynb) |
| 後片付けで「気になる点」 | 出力の `★` の行を読む。所有権なら `host-setup.ps1 -Fix` |

**別のホストへ出すとき**は `-HostName` / `-User` / `-KeyPath` / `-RemotePath` を渡す(新しいホストなら `-AcceptHostKey`)。
手順の全体は [09-new-host](09-new-host.ipynb)。